# Contraste estructural

Bloque «Contraste estructural con TLSP y Martins» del Objetivo 1 (`docs/Contraste_estructural.docx`). Compara la
estructura del dataset v0.2 con la de las instancias públicas de Martins et al. (2023) y del Test Laboratory Scheduling
Problem (TLSP; Mischek y Musliu, 2021): trabajos y tareas por trabajo, tipos de recurso, densidad de elegibilidad y
rangos de duración, y como contexto el horizonte, los plazos, el alistamiento y las perturbaciones
(`src/generador/contraste.py`).

Las unidades se emparejan como en el diccionario de datos (sección 1.5, Tabla 5): muestra y ensayo aquí, trabajo y
operación en Martins et al., proyecto y tarea en el TLSP. Como cada fuente mide el tiempo en otra unidad, las duraciones
se comparan por su dispersión relativa y no por su magnitud.

Necesita el dataset v0.2 (`python -m src.generador.generar --version v0.2`) y las instancias, que no se suben al
repositorio: `python datos/descargar_instancias.py`. Sin los archivos del TLSP, su columna sale de lo que reporta el
artículo. El cuaderno tarda menos de un minuto y guarda en `resultados/tablas/`:

- `contraste_estructural.csv`: la tabla, una fila por medida y una columna por fuente;
- `contraste_estructural_medidas.csv`: las mismas medidas en números;
- `contraste_estructural_instancias.csv`: el detalle por corrida del dataset y por instancia.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.generador import contraste, generar

pd.set_option("display.max_colwidth", 400)
registros, _, resumenes = generar.cargar_version("v0.2")
c = contraste.contrastar(registros, resumenes)
[p.relative_to(RAIZ).as_posix() for p in contraste.guardar(c)]

['resultados/tablas/contraste_estructural.csv',
 'resultados/tablas/contraste_estructural_medidas.csv',
 'resultados/tablas/contraste_estructural_instancias.csv']

## La tabla de contraste

In [2]:
c["tabla"].set_index(["Dimensión", "Medida"])

Este trabajo (dataset v0.2)  \
Dimensión                Medida                                                                                                                                                                                                                       
Trabajos y tareas        Trabajo y tarea                                                                                                                                                 Muestra y ensayo; cada ejecución del ensayo es un registro   
                         Trabajos por instancia                                                       12 873 muestras y 15 388 ensayos en 52 semanas en el escenario base, unas 35 muestras por día (8 897 muestras en el bajo y 19 067 en el alto)   
                         Tareas por trabajo                                                                                                                                   1 a 3 ensayos por muestra (media 1,20; 85 % de las muestras pide uno)   
                         Tipos de trabajo                                                                                                               5 tipos de muestra y 12 métodos; cada tipo pide sus métodos con probabilidades del catálogo   
                         Orden entre las tareas de un trabajo                                                                                                                          Ninguno: los ensayos de una muestra pueden ir a la vez (S28)   
Tipos de recurso         Tipos de recurso                                                                                                                                                                                  Dos: analistas y equipos   
                         Cantidad de recursos                                                                                                                              20 analistas en 3 turnos; 70 equipos en 10 configuraciones de 6 técnicas   
                         Recursos que pide cada tarea                                                                                                                                                 Un analista calificado y un equipo compatible   
                         Ocupación del analista durante la tarea                                             Parcial: en la mediana, 64 % de la duración (43 % a 82 % entre los percentiles 10 y 90); queda libre durante la idoneidad y la corrida   
                         Disponibilidad de los recursos en el tiempo                                                                                          Turnos con rotación, unas 280 ventanas de mantenimiento y 408 fallas de equipo al año   
Densidad de elegibilidad Analistas elegibles por tarea                                                                                                      3 a 15 de 20 (media 11,7): densidad de 59 % pesando por la demanda y de 48 % por método   
                         Equipos elegibles por tarea                                                        2 a 41 de 70 unidades (media 16,7): densidad de 24 % pesando por la demanda y de 11 % por método; por configuración, 1 o 2 de 10 (13 %)   
                         Combinaciones analista y equipo por tarea                                                                                                                                6 a 615 (mediana 121): 17 % de los pares posibles   
                         Cómo se reparte la elegibilidad              Por área, técnica y nivel: un analista de cromatografía no hace fisicoquímica y cada método tiene sus configuraciones de equipo (diseño del laboratorio, secciones 3.1 y 3.3)   
Duraciones               Unidad                                                                                                                                                                                                             Minutos   
                 

## Este trabajo: elegibilidad por método y duración por técnica

Cada ensayo pedido cuenta una vez (primer intento) para la elegibilidad; las duraciones son las de las ejecuciones que
no se interrumpieron, como en el entrenamiento del O2. Escenario base, tres semillas.

In [3]:
tablas = contraste.medir_dataset(registros, resumenes)
pedidos = tablas["pedidos"].query("escenario == 'base'")
lab = contraste.laboratorio()
por_metodo = pedidos.groupby(["metodo", "tecnica"]).agg(
    ensayos=("id_actividad", "size"), analistas=("analistas_calificados", "first"),
    equipos=("equipos_compatibles", "first"), configuraciones=("configuraciones_compatibles", "first"))
por_metodo["densidad de analistas"] = por_metodo["analistas"] / lab["analistas"]
por_metodo["densidad de equipos"] = por_metodo["equipos"] / lab["equipos"]
por_metodo.round(2)

,,ensayos,analistas,equipos,configuraciones,densidad de analistas,densidad de equipos
metodo,tecnica,,,,,,
M01,HPLC,13574,15,41,2,0.75,0.59
M02,HPLC,4573,11,15,1,0.55,0.21
M03,GC,3549,15,4,1,0.75,0.06
M04,GC,8053,11,11,1,0.55,0.16
M05,PSA,3335,11,2,1,0.55,0.03
M06,PSA,5594,9,3,1,0.45,0.04
M07,KF,2224,9,2,1,0.45,0.03
M08,KF,1177,3,2,1,0.15,0.03
M09,DSC,398,11,3,1,0.55,0.04


In [4]:
terminadas = tablas["terminadas"].query("escenario == 'base'")
terminadas.groupby("tecnica").agg(
    ejecuciones=("duracion_simulada", "size"),
    minimo=("duracion_simulada", "min"), p10=("duracion_simulada", lambda s: s.quantile(0.1)),
    mediana=("duracion_simulada", "median"), p90=("duracion_simulada", lambda s: s.quantile(0.9)),
    maximo=("duracion_simulada", "max"), fraccion_analista=("fraccion_analista", "median")).round(2)

,ejecuciones,minimo,p10,mediana,p90,maximo,fraccion_analista
tecnica,,,,,,,
DSC,2003,27.85,66.15,160.10,232.52,299.00,0.40
GC,11845,58.64,104.74,164.64,325.00,398.50,0.54
HPLC,18426,87.09,140.56,246.15,614.41,872.35,0.63
KF,3520,21.89,48.98,83.73,134.64,209.79,0.81
PSA,9133,14.90,39.38,65.81,98.84,144.82,0.78
XRPD,1979,35.73,76.22,123.00,224.72,310.64,0.49


## Martins et al. (2023): lo que traen los archivos

Las 54 instancias de su repositorio. Dos diferencias con el artículo (Tabla 3, p. 192): los trabajos tienen 2 o 3
operaciones, no de 1 a 3, y los tiempos van de 1 a 4, no de 1 a 5. La tabla usa los archivos.

In [5]:
inst, ops = contraste.leer_martins()
por_trabajo = ops.groupby(["instancia", "trabajo"]).size()
pd.Series({"instancias": len(inst), "operaciones": len(ops),
           "operaciones por trabajo": sorted(por_trabajo.unique().tolist()),
           "tiempos de máquina": sorted(ops["duracion"].unique().tolist()),
           "fracción del analista": sorted(ops["fraccion_analista"].round(3).unique().tolist()),
           "liberación y vencimiento distintos de 0": int((ops[["liberacion", "vencimiento"]] != 0).any(axis=1).sum()),
           "todos los trabajos de un tipo son iguales": contraste._tipos_iguales(ops)})

instancias                                                     54
operaciones                                                  3897
operaciones por trabajo                                    [2, 3]
tiempos de máquina                           [1.0, 2.0, 3.0, 4.0]
fracción del analista                          [0.25, 0.267, 0.3]
liberación y vencimiento distintos de 0                         0
todos los trabajos de un tipo son iguales                    True
dtype: object

In [6]:
ops.groupby(["flexibilidad", "analistas"])[["analistas_elegibles", "maquinas_elegibles", "densidad_analistas",
                                            "densidad_maquinas"]].mean().round(2)

analistas_elegibles  maquinas_elegibles  \
flexibilidad analistas                                            
0.3          2                         1.06                2.14   
             3                         1.20                1.99   
             7                         2.38                2.38   
0.6          2                         1.31                4.28   
             3                         1.73                4.36   
             7                         4.13                4.42   

                        densidad_analistas  densidad_maquinas  
flexibilidad analistas                                         
0.3          2                        0.53               0.31  
             3                        0.40               0.28  
             7                        0.34               0.34  
0.6          2                        0.65               0.61  
             3                        0.58               0.62  
             7                        0.59               0.63

## TLSP (Mischek y Musliu, 2021)

Las tres instancias reales que describe el artículo (Tabla 4, p. 551), con la densidad de elegibilidad que sale de sus
promedios: empleados calificados por trabajo sobre empleados, y mesas disponibles por trabajo sobre mesas.

In [7]:
contraste.TLSP_REALES.round(2)

,instancia,proyectos,trabajos,franjas,empleados,mesas,grupos_de_equipos,empleados_calificados_por_trabajo,mesas_por_trabajo,densidad_empleados,densidad_mesas
0,2019-04,74,297,606,22,17,1,5.49,3.06,0.25,0.18
1,2019-07,59,251,700,24,17,1,5.33,3.17,0.22,0.19
2,2019-10,59,223,572,19,17,1,5.70,3.48,0.30,0.20
